# PayGuard — Model 3: XGBoost

Supervised, **boosting** ensemble — trees built sequentially, each one correcting the errors of the previous tree. Usually the strongest performer on tabular fraud data, at the cost of being more sensitive to tuning than Random Forest.

If not already installed:
```
pip install xgboost
```

In [1]:
import joblib
import os
import pandas as pd
from xgboost import XGBClassifier
from evaluation import evaluate_model, print_comparison_table

bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]
X_train_smote, y_train_smote = bundle["X_train_smote"], bundle["y_train_smote"]
X_train_under, y_train_under = bundle["X_train_under"], bundle["y_train_under"]

print("Bundle loaded. Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

Bundle loaded. Train shape: (64000, 21) | Test shape: (16000, 21)


## Variant 1: Class-Weighted (via `scale_pos_weight`)

XGBoost doesn't take a `class_weight` dict like scikit-learn models — instead it takes a single scalar `scale_pos_weight`, the ratio of negative to positive examples. This is XGBoost's own equivalent of what we computed in notebook 3 as `class_weight_dict`, just expressed differently.

In [2]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_weighted = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss", random_state=42, n_jobs=-1,
)
xgb_weighted.fit(X_train_scaled, y_train)

y_pred_weighted = xgb_weighted.predict(X_test_scaled)
y_score_weighted = xgb_weighted.predict_proba(X_test_scaled)[:, 1]

result_weighted = evaluate_model(y_test, y_pred_weighted, "XGBoost", "Class-Weighted", y_score_weighted)

scale_pos_weight: 358.55

  XGBoost (Class-Weighted)
Precision: 0.7292   Recall: 0.7778   F1: 0.7527

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15942                  13
Actual Fraud                  10                  35

ROC-AUC: 0.9723

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00     15955
       Fraud       0.73      0.78      0.75        45

    accuracy                           1.00     16000
   macro avg       0.86      0.89      0.88     16000
weighted avg       1.00      1.00      1.00     16000



## Variant 2: SMOTE

In [3]:
xgb_smote = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric="logloss", random_state=42, n_jobs=-1,
)
xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = xgb_smote.predict(X_test_scaled)
y_score_smote = xgb_smote.predict_proba(X_test_scaled)[:, 1]

result_smote = evaluate_model(y_test, y_pred_smote, "XGBoost", "SMOTE", y_score_smote)


  XGBoost (SMOTE)
Precision: 0.7200   Recall: 0.8000   F1: 0.7579

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15941                  14
Actual Fraud                   9                  36

ROC-AUC: 0.9843

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00     15955
       Fraud       0.72      0.80      0.76        45

    accuracy                           1.00     16000
   macro avg       0.86      0.90      0.88     16000
weighted avg       1.00      1.00      1.00     16000



## Variant 3: Random Undersampling

In [4]:
xgb_under = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    eval_metric="logloss", random_state=42, n_jobs=-1,
)
xgb_under.fit(X_train_under, y_train_under)

y_pred_under = xgb_under.predict(X_test_scaled)
y_score_under = xgb_under.predict_proba(X_test_scaled)[:, 1]

result_under = evaluate_model(y_test, y_pred_under, "XGBoost", "Undersampling", y_score_under)


  XGBoost (Undersampling)
Precision: 0.0289   Recall: 0.8889   F1: 0.0559

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           14610                1345
Actual Fraud                   5                  40

ROC-AUC: 0.9539

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.92      0.96     15955
       Fraud       0.03      0.89      0.06        45

    accuracy                           0.92     16000
   macro avg       0.51      0.90      0.51     16000
weighted avg       1.00      0.92      0.95     16000



## Compare the three preprocessing variants for this model

In [5]:
xgb_results = [result_weighted, result_smote, result_under]
xgb_comparison_df = print_comparison_table(xgb_results)


  MODEL COMPARISON (sorted by F1)
  model        variant  precision  recall     f1  roc_auc  true_positives  false_positives  true_negatives  false_negatives
XGBoost          SMOTE     0.7200  0.8000 0.7579   0.9843              36               14           15941                9
XGBoost Class-Weighted     0.7292  0.7778 0.7527   0.9723              35               13           15942               10
XGBoost  Undersampling     0.0289  0.8889 0.0559   0.9539              40             1345           14610                5


## Feature importance

Same idea as Random Forest's importances — worth comparing all three models' top features side by side in your report.

In [6]:
importance_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "importance": xgb_weighted.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
10,hour_of_day,0.161012
1,receiver_is_merchant,0.160369
16,mode_CASH_IN,0.098041
12,sender_balance_error,0.094263
14,balance_ratio,0.091933
8,amount_deviation_ratio,0.065568
0,amount,0.063832
3,sender_balance_after,0.056904
13,receiver_balance_delta,0.052385
2,sender_balance_before,0.041088


## Save the best variant's model + results

In [7]:
os.makedirs("../models", exist_ok=True)

best_variant = xgb_comparison_df.iloc[0]
print(f"Best variant for XGBoost: {best_variant['variant']} (F1={best_variant['f1']}, recall={best_variant['recall']})")

variant_to_model = {"Class-Weighted": xgb_weighted, "SMOTE": xgb_smote, "Undersampling": xgb_under}
best_model = variant_to_model[best_variant["variant"]]

joblib.dump(best_model, "../models/xgboost.pkl")
xgb_comparison_df.to_csv("../models/xgboost_results.csv", index=False)
print("Saved model to ../models/xgboost.pkl")
print("Saved results to ../models/xgboost_results.csv")

Best variant for XGBoost: SMOTE (F1=0.7579, recall=0.8)
Saved model to ../models/xgboost.pkl
Saved results to ../models/xgboost_results.csv


## Next: `04d_model_isolation_forest.ipynb`

This is where things change: Isolation Forest is **unsupervised**, so it trains without `y_train` at all, and only ever uses `X_train_scaled` (no SMOTE/undersampling/class-weight variants — those only make sense for supervised models).